# n002 — Graph Analysis

This notebook analyses the **social interaction graphs** produced by each experiment run. Graphs are pre-computed and stored as `graph.pkl` per run.

**REQUIRES:** `002_make_graph.py` to be run first.

**Outputs** (saved to `<ROOT>/plots/`):
- `002_graph_global_metrics.pdf` — grouped boxplots for global graph metrics (density, reciprocity, …)
- `002_graph_global_communities.pdf` — strip plot of community-level metrics weighted by community size
- `002_graph_time_metrics.pdf` — time-series of graph metrics over the course of each run

**Graph data structure** (`graph.pkl`):
- `graph_data['graph_data']` — dict with keys:
  - `density`, `reciprocity`, `positive_reciprocity`, `negative_reciprocity` — scalar global metrics
  - `community_summary` — per-community stats (`size`, `intra_weight`, `out_weight`, `intra_share`)
  - `time_metrics` — DataFrame with one row per timestep for the time-evolving metrics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from core.utils.analysis_utils import get_exp_folders, get_last_ts
from core.utils import ROOT
from collections import defaultdict
import pickle as pkl
import pandas as pd
from matplotlib import colors as mcolors
import itertools
from tqdm import tqdm

# force reload these everytime
import importlib
importlib.reload(importlib.import_module('analysis_scripts.plot_utils'))
from analysis_scripts.plot_utils import color_map, plot_params

plt.rcParams.update(plot_params)


## Configuration

Set `EXP_BASE_NAMES` to the list of experiment condition names to compare. `SHOW_STD` is loaded but not currently used in this notebook (all time-series plots show ±1 std bands unconditionally).

In [ ]:
EXP_BASE_NAMES = [] # TO DEFINE

# Change this if you want other names in the plot
PLOT_NAMES = {exp_name: exp_name.replace("_", " ").title() for exp_name in EXP_BASE_NAMES}

EXP_PATH = ROOT / "logs"
SHOW_STD = False

experiments = {}
for base_name in EXP_BASE_NAMES:
    experiments[base_name] = get_exp_folders(log_path=EXP_PATH, 
                                             exp_name=base_name, 
                                             only_completed=True)

## Data Loading

Loads `graph.pkl` for every completed run and records the experiment length from `open_gridworld.log`. Results are stored in:
- `graph_data[base_name]` — list of pkl dicts, one per run
- `exp_lengths[base_name]` — list of final timesteps (not currently used downstream, available for normalisation)

In [ ]:
graph_data = defaultdict(list)
exp_lengths = defaultdict(list)
for base_name in experiments:
    for exp in tqdm(experiments[base_name], desc=base_name):
        exp_path = EXP_PATH / exp
        exp_data = pkl.load(open(exp_path / "graph.pkl", "rb"))
        graph_data[base_name].append(exp_data)
        last_ts = get_last_ts(exp_path / 'open_gridworld.log')
        exp_lengths[base_name].append(last_ts)

## Global Graph Metrics — Grouped Boxplots

Builds a tidy long-format DataFrame from four scalar metrics per run:
- **Density** — fraction of possible edges that exist
- **Reciprocity** — fraction of edges that are mutual
- **Positive Reciprocity** — reciprocity restricted to positive-weight edges
- **Negative Reciprocity** — reciprocity restricted to negative-weight edges

Each metric group is shown as a set of side-by-side boxplots, one per condition. Boxes are translucent with full-opacity medians; a white dot marks the mean. Vertical grey lines separate metric groups.

Saved as `002_graph_global_metrics.pdf`.

In [ ]:
metrics = ['density', 'reciprocity', 'positive_reciprocity', 'negative_reciprocity']
metric_labels = [m.replace('_', ' ').title() for m in metrics]
label_map = dict(zip(metrics, metric_labels))
palette = [color_map[b] for b in EXP_BASE_NAMES]

# ---- long-form dataframe
rows = []
for base_name in EXP_BASE_NAMES:
    for exp_data in graph_data[base_name]:
        gd = exp_data['graph_data']
        for m in metrics:
            vals = np.asarray(gd[m]).ravel()
            for v in vals:
                rows.append({
                    "experiment": base_name,
                    "metric": label_map[m],
                    "value": float(v),
                })
df = pd.DataFrame(rows)

# ---- plot with matplotlib
fig, ax = plt.subplots(figsize=(8, 5))

# collect per-metric and per-experiment data
positions = []
data_groups = []
colors = []

group_width = 0.7
n_experiments = len(EXP_BASE_NAMES)
offset = np.linspace(-group_width/2, group_width/2, n_experiments)

for i, metric in enumerate(metric_labels):
    df_m = df[df["metric"] == metric]
    for j, exp in enumerate(EXP_BASE_NAMES):
        vals = df_m[df_m["experiment"] == exp]["value"].dropna().values
        pos = i + offset[j]
        positions.append(pos)
        data_groups.append(vals)
        colors.append(palette[j])

bp = ax.boxplot(
        data_groups,
        positions=positions,
        patch_artist=True,
        showfliers=False,
        showcaps=False,
        whis=1,
        medianprops={"linewidth": 2},
        whiskerprops={"linewidth": 1},
        capprops={"linewidth": 1},
        widths=group_width / n_experiments * 0.9,
    )

for x, vals in zip(positions, data_groups):
        mean_val = np.mean(vals)
        ax.plot(
            x, mean_val,
            marker='o',
            markersize=6,
            markerfacecolor='white',
            markeredgecolor='black',
            markeredgewidth=1.3,
            zorder=5
        )

# boxes: translucent fill, no borders
for patch, c in zip(bp["boxes"], colors):
    r, g, b = mcolors.to_rgb(c)
    patch.set_facecolor((r, g, b, 0.7))
    patch.set_edgecolor("none")

# medians: full-opacity in same hue
for med, c in zip(bp["medians"], colors):
    r, g, b = mcolors.to_rgb(c)
    med.set_color((r, g, b, 1.0))
    med.set_linewidth(1.5)

# whiskers/caps: color-match per box (two per box)
rep_colors = list(itertools.chain.from_iterable([[c, c] for c in colors]))

for whisk, c in zip(bp["whiskers"], rep_colors):
    r, g, b = mcolors.to_rgb(c)
    whisk.set_color((r, g, b, 1.0))

for cap, c in zip(bp["caps"], rep_colors):
    r, g, b = mcolors.to_rgb(c)
    cap.set_color((r, g, b, 1.0))

# formatting
ax.set_xticks(range(len(metric_labels)))
ax.set_xticklabels(metric_labels)
ax.set_ylabel("Value")
ax.set_title("Interaction graph metrics")
ax.yaxis.grid(alpha=0.3)

for i in range(len(metrics) - 1):
    ax.axvline(x=i + 0.5, color="gray", linewidth=1, alpha=0.5)

# ensure all boxes visible
ax.set_xlim(-0.5, len(metrics) - 0.5)

# Add legend
legend_handles = [plt.Rectangle((0,0),1,1, facecolor=color_map[base_name], alpha=0.7) 
                 for base_name in EXP_BASE_NAMES]
ax.legend(legend_handles, PLOT_NAMES.values(), loc='upper right')

plt.tight_layout()
plt.savefig(ROOT / 'plots' / '002_graph_global_metrics.pdf', dpi=300)
plt.show()

## Community Metrics — Strip Plot

Visualises per-community statistics pooled across all runs, with **point size proportional to community size**.

Metrics shown (each normalised to [0, 1] across all communities before plotting):
- **Intra Weight** — total weight of edges within the community
- **Out Weight** — total weight of edges leaving the community
- **Intra Share** — fraction of a community's total edge weight that stays internal

Jitter is added to x-positions to reduce overplotting. Normalisation is done independently per metric so different scales are visually comparable.

Saved as `002_graph_global_communities.pdf`.

In [ ]:
# Create strip plot for community metrics with community size as point size (normalized)
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
community_metrics = ['intra_weight', 'out_weight', 'intra_share']
metric_labels = [metric.replace('_', ' ').title() for metric in community_metrics]

all_community_data = []
for base_name in graph_data:
    for exp_data in graph_data[base_name]:
        if 'community_summary' in exp_data['graph_data']:
            community_data = exp_data['graph_data']['community_summary']
            n_communities = len(community_data.get('intra_weight', []))
            
            for i in range(n_communities):
                community_info = {
                    'base_name': base_name,
                    'community_size': community_data['size'][i],
                }
                
                # Add all metrics for this community
                for metric in community_metrics:
                    if metric in community_data and i < len(community_data[metric]):
                        community_info[metric] = community_data[metric][i]
                
                all_community_data.append(community_info)


# Normalize each metric to 0-1 range
for metric in community_metrics:
    all_values = [d[metric] for d in all_community_data if metric in d]
    if all_values:
        min_val = min(all_values)
        max_val = max(all_values)
        value_range = max_val - min_val
        
        # Normalize values to 0-1
        for d in all_community_data:
            if metric in d:
                if value_range > 0:
                    d[f'{metric}_normalized'] = (d[metric] - min_val) / value_range
                else:
                    d[f'{metric}_normalized'] = 0.5  # If all values are the same

# Set up positions for strip plots
n_metrics = len(community_metrics)
n_base_names = len(EXP_BASE_NAMES)
width = 0.7 / n_base_names

for i, metric in enumerate(community_metrics):
    for j, base_name in enumerate(EXP_BASE_NAMES):
        # Filter data for this base_name and metric
        base_data = [d for d in all_community_data if d['base_name'] == base_name and f'{metric}_normalized' in d]
        
        if base_data:
            # Position for this group: metric position + offset for base_name
            position = i + (j - (n_base_names-1)/2) * width
            
            # Add some random jitter to x positions to avoid overlap
            x_positions = [position + np.random.normal(0, 0.02) for _ in base_data]
            y_values = [d[f'{metric}_normalized'] for d in base_data]
            sizes = [20 + d['community_size'] * 3 for d in base_data]  # Scale for visibility
            
            ax.scatter(x_positions, y_values, 
                      s=sizes, 
                      c=color_map[base_name], 
                      alpha=0.6, 
                      label=base_name if i == 0 else "",  # Only show legend for first metric
                      edgecolors='black',
                      linewidth=0.5)

# Set labels and formatting
ax.set_xticks(range(n_metrics))
ax.set_xticklabels(metric_labels)
ax.set_ylabel('Normalized Metric Values (0-1)')
ax.set_title('Distribution of Normalized Community Metrics\n(Point size = Community size)')
ax.set_ylim(-0.05, 1.05)  # Set y-axis limits to show full 0-1 range

ax.yaxis.grid(alpha=0.3)

for i in range(len(community_metrics) - 1):
    ax.axvline(x=i + 0.5, color="gray", linewidth=1, alpha=0.5)

# ensure all boxes visible
ax.set_xlim(-0.5, len(community_metrics) - 0.5)

# Add legend
legend_handles = [plt.Rectangle((0,0),1,1, facecolor=color_map[base_name], alpha=0.7) 
                 for base_name in EXP_BASE_NAMES]
ax.legend(legend_handles, PLOT_NAMES.values(), loc='lower right')

plt.tight_layout()
plt.savefig(ROOT / 'plots' / '002_graph_global_communities.pdf', dpi=300)
plt.show()

## Time-Series Metrics

Plots 8 graph metrics over time, one subplot per metric. Each curve is the mean across runs; the shaded band is ±1 std. A dot marks the end of each individual run (where the series terminates).

Metrics tracked (see the **Explanation of metrics** cell below for definitions):
`modularity`, `partition_entropy`, `communities`, `neg_polarization`, `avg_clustering`, `avg_degree`, `mean_participation`, `entropy`

Shorter runs are padded with `NaN` before aggregation so means are computed only over available data.

Saved as `002_graph_time_metrics.pdf`.

In [ ]:
metrics_to_plot = ['modularity', 'partition_entropy', 'communities', 
                   'neg_polarization', 'avg_clustering', 'avg_degree', 
                   'mean_participation', 'entropy']

fig, axes = plt.subplots(len(metrics_to_plot), 1, figsize=(7, 3 * len(metrics_to_plot)), sharex=True)
if len(metrics_to_plot) == 1:
    axes = [axes]
    
for metric, ax in zip(metrics_to_plot, axes):
    for base_name, data in graph_data.items():
        lengths = []
        values = []
        for exp in data:
            exp_results = exp['graph_data']['time_metrics'][metric].values
            values.append(exp_results)
            lengths.append(len(exp_results))
        # Pad sequences to the same length
        padded = np.full((len(values), max(lengths)), np.nan)
        for i, v in enumerate(values):
            padded[i, :len(v)] = v
        mean_values = np.nanmean(padded, axis=0)
        std_values = np.nanstd(padded, axis=0)

        ax.plot(mean_values, label=base_name, color=color_map[base_name])
        ax.fill_between(range(len(mean_values)), 
                        mean_values - std_values, 
                        mean_values + std_values, 
                        color=color_map[base_name], alpha=0.3)
        exp_ends = [[t-1, mean_values[t-1]] for t in lengths]
        ax.scatter(*zip(*exp_ends), color=color_map[base_name], marker='o')
    # ax.set_title(f'{metric.replace("_", " ").title()} Over Time')
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.legend()
    ax.grid(True, alpha=0.3)

ax.set_xlabel('Time Steps')
plt.tight_layout()
plt.savefig(ROOT / 'plots' / '002_graph_time_metrics.pdf', dpi=300)
plt.show()
        


## Explanation of metrics
**Modularity**

Modularity Q measures how strongly the graph decomposes into communities relative to a null model with the same node strengths (weighted degrees).

- High Q (≈ 0.3–0.8 in practice): strong within-community concentration.
- Low/near 0: structure close to random given degrees.
- Negative: fewer within-community edges than expected.

**Partition Entropy**

It measures how evenly community sizes are distributed in the partition: high values mean many similarly sized communities, low values mean one or few dominant ones.
H reflects *diversity/balance of community sizes* (not their separation).
- Higher H: many communities with balanced sizes (more differentiated roles).
- Lower H: one dominant block or very skewed sizes.

The normalized version is scaled by the log(number of communities)

**Participation**

It measures how evenly each node distributes its links across communities
- P ~ 0: Node connects almost exclusively within its own community (specialist).
- P high (approaches 1): node spreads links across many communities (broker).

**Negative cuts and polarization**

Compute statistics of negative-weight edges that cross communities.
It measures how much negative interaction in the graph occurs between communities, and whether that antagonism is concentrated between just one pair of communities (polarization ≈ 1) or spread more evenly across many community boundaries (polarization ≈ 0).

**Average degree**

Average degree of connectivity in the graph. It measures connection density. Higher average degree indicates more locally complex connectivity.

**Average clustering**

Measures the average clustering of the graph. 

**Entropy**

Captures heterogeneity of node connectivity; higher entropy → more irregular structure

